In [ ]:
import pandas as pd
import re
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
df = pd.read_csv('IMDB Dataset.csv')
df.head()

In [ ]:
df.shape, df['sentiment'].value_counts(), df.isnull().sum()

In [ ]:
stemmer = PorterStemmer()
stops = ENGLISH_STOP_WORDS - {'not', 'no', 'nor'}

def clean(text):
    words = re.findall(r'[a-z]+', text.lower())
    return ' '.join(stemmer.stem(w) for w in words if w not in stops)

df['clean_review'] = df['review'].fillna('').apply(clean)

In [ ]:
df[['review', 'clean_review']].head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df['clean_review'], df['sentiment'], test_size=0.2, random_state=42, stratify=df['sentiment'])

In [ ]:
tfidf = TfidfVectorizer(max_features=10000)
X_train = tfidf.fit_transform(X_train)
X_test = tfidf.transform(X_test)
X_train.shape

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [ ]:
print('Accuracy:', accuracy_score(y_test, y_pred))
print('F1-score:', f1_score(y_test, y_pred, pos_label='positive'))

In [ ]:
weights = pd.Series(model.coef_[0], index=tfidf.get_feature_names_out())
print('Positive words:')
print(weights.nlargest(10))
print('Negative words:')
print(weights.nsmallest(10))

## Final Summary

Loaded the IMDB movie reviews, cleaned the text using lowercase, punctuation removal, stopword removal and stemming, then split the data into training and testing sets. Converted reviews into TF-IDF features and trained a Logistic Regression model to predict positive or negative sentiment. Evaluated the predictions using Accuracy and F1-score and displayed the top 10 predictive words for each class.